# Model fitting notebook
---

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.optimize
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.special import logsumexp
import warnings
from scipy.optimize import curve_fit
from scipy.optimize import approx_fprime
import scipy.stats as stats
from pybads import BADS

from matplotlib.colors import LinearSegmentedColormap

# TEMP
warnings.simplefilter("ignore", category=FutureWarning)

from src.src import *

# Set some defaults
plt.rc("axes.spines", top=False, right=False)
plt.rcParams["mathtext.default"] = "regular"
sns.set_theme(font_scale=1.2)
sns.set_style("ticks")
sns.set_palette('deep')

%config InlineBackend.figure_format = "retina"

---
## Load in data

In [2]:
# Read in data to be fit
df = pd.read_csv("combined-data.csv")
df.head(10)

,si,SN,id,tester,TN,move_cycle,hX,hY,rot_hX,rot_hY,...,pert_trial,x_f_z,x_f_outlier,no_fit,x_f_mean,x_f_clean,adaptation,sigma_motor_adapt,pert_resp,sigma_motor_sdt
0,1,1,01_i,s,101,101,-6.523699,95.708669,0.0,0.0,...,True,-1.121559,False,False,1.541759,-1.606234,0.983717,2.246088,NaN,NaN
1,1,1,01_i,s,102,102,-0.073699,96.518669,0.0,0.0,...,False,0.036549,False,False,1.541759,1.624840,0.000000,2.246088,NaN,NaN
2,1,1,01_i,s,103,103,-3.833699,93.848669,0.0,0.0,...,True,-0.591950,False,False,1.541759,-0.128646,0.581988,2.246088,NaN,NaN
3,1,1,01_i,s,104,104,0.466301,92.658669,0.0,0.0,...,False,0.245150,False,False,1.541759,2.206829,0.000000,2.246088,NaN,NaN
4,1,1,01_i,s,105,105,-0.043699,90.888669,0.0,0.0,...,True,-0.011512,False,False,1.541759,1.490751,0.870253,2.246088,NaN,NaN
5,1,1,01_i,s,106,106,2.096301,91.888669,0.0,0.0,...,False,0.557073,False,False,1.541759,3.077081,0.000000,2.246088,NaN,NaN
6,1,1,01_i,s,107,107,0.256301,94.218669,0.0,0.0,...,True,-0.305145,False,False,1.541759,0.671527,-0.190481,2.246088,NaN,NaN
7,1,1,01_i,s,108,108,0.906301,95.168669,0.0,0.0,...,False,0.488799,False,False,1.541759,2.886601,0.000000,2.246088,NaN,NaN
8,1,1,01_i,s,109,109,2.026301,93.148669,0.0,0.0,...,True,1.663108,False,False,1.541759,6.162874,-5.025827,2.246088,NaN,NaN
9,1,1,01_i,s,110,110,-4.003699,93.218669,0.0,0.0,...,False,-1.312598,False,False,1.541759,-2.139226,0.000000,2.246088,NaN,NaN


---
## Helper functions

---
## Fit data using MLE

In [3]:
fit_single_subj = False

numtrials_adapt = 1540
numtrials_sdt = 800
numtrials_both = numtrials_adapt + numtrials_sdt

# We switch off the printing from BADS
options = {"display" : 'off'}

fit_task = "both"
n_sims = 1000
n_inits = 5

if fit_single_subj:
    # Pick your subject here
    subjects = np.array([14])
else:
    subjects = df['SN'].unique()


In [ ]:
%%time
# PIECE parameter bounds: alpha, beta, sigma_p, sigma_pert, b,
bounds       = ((0.1, 5), (0.001, 1), (0.1, 10), (0.1, 120), (-5, 5))
plaus_bounds = ((0.5, 4), (0.01, 0.5), (0.3, 6), (3, 20), (-2.5, 2.5))

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_piece, aic, bic = fit_bads("piece", s, df, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, fit_task=fit_task, options=options)
    results_list.append(fits_piece)
    aic_list.append(aic)
    bic_list.append(bic)

best_params_piece = np.array([res.x for res in results_list]).squeeze()

fits_piece = pd.DataFrame({
    'SN': subjects,
    'model': "piece",
    "fit_task": fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
fits_piece[["alpha", "beta", "sigma_p", "sigma_pert", "b"]] = pd.DataFrame(fits_piece['params'].tolist(), index=fits_piece.index)

if fit_task == "adapt":
    fits_piece_adapt = fits_piece.copy()
elif fit_task == "both":
    fits_piece_both = fits_piece.copy()
elif fit_task == "sdt": 
    fits_piece_sdt = fits_piece.copy()

# # Checks path and creates new csv
# save_csv(fits_piece, model='piece')
# fits_piece.to_csv("model-fits/nolapse_piece_simult.csv", index=False)
# fits_piece.to_csv("model-fits/nolapse_piece_adapt.csv", index=False)


In [4]:
%%time
# Fit VCO
if fit_task == "sdt":
    # Parameter bounds (C, sigma_v, b) 
    bounds       = [(0.1, 10), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.5, 6), (-2.5, 2.5)]
else:
    # Parameter bounds (C, B, sigma_v, b) 
    bounds       = [(0.1, 10), (0.01, 1), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.05, 0.6), (0.5, 6), (-2.5, 2.5)]

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_vco, aic, bic = fit_bads("vco", s, df, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, 
                                  fit_task=fit_task, options=options)
    results_list.append(fits_vco)
    aic_list.append(aic)
    bic_list.append(bic)

best_params_vco = np.array([res.x for res in results_list]).squeeze()

fits_vco = pd.DataFrame({
    'SN': subjects,
    'model': "vco",
    "fit_task": fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
if fit_task == "adapt":
    fits_vco[["C", "B", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_adapt = fits_vco.copy()
elif fit_task == "both":
    fits_vco[["C", "B", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_both = fits_vco.copy()
elif fit_task == "sdt": 
    fits_vco[["C", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_sdt = fits_vco.copy()

# # Checks path and creates new csv
# fits_vco.to_csv("model-fits/SAVE_vco_sdt.csv", index=False)
# save_csv(fits_vco, model='vco')
# fits_vco.to_csv("model-fits/fixed_vco_simult.csv", index=False)
# fits_vco.to_csv("model-fits/fixed_vco_adapt.csv", index=False)


Subject Number: 1
Running optimization 0...


/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


Subject Number: 1
Running optimization 1...
Subject Number: 1
Running optimization 2...


/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


Subject Number: 1
Running optimization 3...


/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


... (43 outputs left out)

/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


Subject Number: 16
Running optimization 4...


/opt/anaconda3/envs/ige-ege/lib/python3.11/site-packages/pybads/init_functions/init_sobol.py:39: RuntimeWarning: invalid value encountered in cast
  str_seed = u0[0 : np.minimum(11, len(u0))].astype(np.uint64)


CPU times: user 2h 22min 19s, sys: 4h 9min 18s, total: 6h 31min 38s
Wall time: 52min 32s


In [ ]:
%%time
# Fit CO model
if fit_task == "sdt":
    # Parameter bounds (C, sigma_comb) 
    bounds = [(0.1, 10), (0.1, 10.0)]
    plaus_bounds = [(0.5, 5), (0.5, 6)]
else:
    # Parameter bounds (C, B, sigma_comb, b) 
    bounds       = [(0.1, 10), (0.01, 1), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.05, 0.6), (0.5, 6), (-2.5, 2.5)]

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_co, aic, bic = fit_bads("co", s, df, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, fit_task=fit_task, options=options)
    results_list.append(fits_co)
    aic_list.append(aic)
    bic_list.append(bic)

fits_co = pd.DataFrame({
    'SN': subjects,
    'model': "co",
    "fit_task" : fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
if fit_task == "adapt":  
    fits_co[["C", "B", "sigma_comb", "b"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_adapt = fits_co.copy()
elif fit_task == "both":
    fits_co[["C", "B", "sigma_comb", "b"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_both = fits_co.copy()
elif fit_task == "sdt": 
    fits_co[["C", "sigma_comb"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_sdt = fits_co.copy()

# # Checks path and creates new csv
# save_csv(fits_co, model='co')
# fits_co.to_csv("model-fits/nolapse_co_simult.csv", index=False)
# fits_co.to_csv("model-fits/nolapse_co_adapt.csv", index=False)
